# GeLB intro: 2-D Taylor-Green vortex decay

Welcome — this notebook compiles and runs a complete 2-D Lattice-Boltzmann simulation end-to-end in Colab, using **[GeLB](https://github.com/dchirila/gelb)** to generate the Fortran source from a ~70-line high-level description.

You will:

1. Install GeLB and its runtime dependencies (`gfortran`, NetCDF-Fortran).
2. Write a Taylor-Green vortex problem as a `.gelb` source file.
3. Compile it with `gelbc` and build the generated Fortran with `gfortran`.
4. Run the simulation.
5. Load the NetCDF output and visualise the kinetic-energy decay against the analytical solution.

Expected runtime: about **2 minutes** end-to-end — most of it is the one-time `apt-get` step.

*This notebook runs on a free Colab CPU runtime. For the GPU path (CUDA C++, 3-D), see the companion [`gelb_gpu_3d.ipynb`](./gelb_gpu_3d.ipynb).*


## 1. Install runtime dependencies

The Fortran code that `gelbc` emits depends on:

- **`gfortran`** — the Fortran compiler. Anything ≥ 9 works.
- **`libnetcdff-dev`** — the Fortran bindings to NetCDF, used by the generated code for structured file output.


In [ ]:
!apt-get update -qq
!apt-get install -y -qq gfortran libnetcdff-dev > /dev/null
!gfortran --version | head -1

## 2. Install GeLB

GeLB 1.0.0, the release described in the SoftwareX paper, installs from its tag on GitHub (about 30 s):

In [ ]:
!pip install -q "git+https://github.com/dchirila/gelb.git@v1.0.0"

# gelb-run starts the compiler and its progress monitor through a Python
# interpreter that can import gelb: name this notebook's own.
import os, sys
os.environ["GELB_PYTHON"] = sys.executable

In [ ]:
# Verify the install worked. Expect 'gelbc', 'gelb-run' and 'gelb-view' on PATH.
!which gelbc gelb-run gelb-view
!python -c "import importlib.metadata as m; print('gelb', m.version('gelb'))"

GeLB source is shown with syntax highlighting by the Pygments lexer that the GeLB manuals use:

In [ ]:
!curl -sSfL -o gelb_lexer.py https://raw.githubusercontent.com/dchirila/gelb/v1.0.0/docs/_ext/gelb_lexer.py

from gelb_lexer import GelbLexer
from pygments import highlight
from pygments.formatters import HtmlFormatter
from IPython.display import HTML, display

def show_gelb(path, max_height=None):
    """Display a .gelb file with syntax highlighting and line numbers."""
    with open(path) as f:
        html = highlight(f.read(), GelbLexer(),
                         HtmlFormatter(noclasses=True, style="friendly", linenos="inline",
                                       prestyles="font-family: monospace; font-size: 13px; line-height: 125%;"))
    box = f"max-height: {max_height}px; overflow: auto;" if max_height else ""
    display(HTML(f'<div style="{box}">{html}</div>'))

## 3. Write the GeLB program

The entire 2-D Taylor-Green vortex is described below in **~70 lines**. Notice:

- `lattices` declares the data layout (D2Q9 = 2 dimensions, 9 velocities).
- `function` declares side-effect-free helpers (`Macro`, `Eq`) that the compiler inlines.
- `initializer`, `dynamic`, and `gauge` are the three phase blocks; `node_classes` attaches them to the fluid region.
- `geometry` paints the domain — here a fully-periodic uniform box.

The cell below writes the program to `taylor_green.gelb` and shows it with syntax highlighting. To change the program, double-click the cell's title to open its source, edit it, and run the cell again.


In [ ]:
#@title The GeLB program `taylor_green.gelb` (double-click to edit)
GELB_SOURCE = r'''
// 2D Taylor-Green vortex decay — D2Q9 BGK LBM
//
// Standard benchmark for incompressible Navier-Stokes solvers.
// Analytical initial condition decays exponentially with rate nu*k^2.
// Uses user-defined functions (Macro, Eq) for code reuse.

simulation_description("2D Taylor-Green vortex (LBM-BGK, D2Q9)");

global_constants {
    const int N = 64;
    const real NU = 1.0 / 6.0;
    const real TAU_INV = 2.0 / (6.0 * NU + 1.0);
    const real U_MAX = 0.04;
    const real PI = 3.141592653589793;
    const real K = 2.0 * PI / N;

    // D2Q9 lattice velocities
    const array[real, 9] EX = {0.0, 1.0, 0.0, -1.0,  0.0, 1.0, -1.0, -1.0,  1.0};
    const array[real, 9] EY = {0.0, 0.0, 1.0,  0.0, -1.0, 1.0,  1.0, -1.0, -1.0};
    // D2Q9 weights
    const array[real, 9] W = {4.0/9.0,
        1.0/9.0, 1.0/9.0, 1.0/9.0, 1.0/9.0,
        1.0/36.0, 1.0/36.0, 1.0/36.0, 1.0/36.0};
}

lattices {
    lattice f(2, 9, real, {N, N}, {true, true});
}

function Macro(array[real, 9] fIn) -> (real rho, real ux, real uy) {
    rho = sum(fIn);
    ux = dot_prod(EX, fIn) / rho;
    uy = dot_prod(EY, fIn) / rho;
}

function Eq(real rho, real ux, real uy) -> (array[real, 9] feq) {
    real usq = ux*ux + uy*uy;
    array[real, 9] eu = EX * ux + EY * uy;
    feq = W * rho * (1.0 + 3.0*eu + 4.5*eu*eu - 1.5*usq);
}

initializer InitTG {
    real xc = x + 0.5;
    real yc = y + 0.5;

    // Taylor-Green analytical fields at t=0
    real rho = 1.0 - 0.75 * U_MAX^2 * (cos(2.0*K*xc) + cos(2.0*K*yc));
    real ux = -1.0 * U_MAX * cos(K*xc) * sin(K*yc);
    real uy = U_MAX * sin(K*xc) * cos(K*yc);
    f.out[x, y | :] = Eq(rho, ux, uy);
}

dynamic BGKCollStream {
    // Pull streaming + collision
    array[real, 9] fIn = upstream_dfs(f, x, y, {EX, EY});

    real rho;
    real ux;
    real uy;

    {rho, ux, uy} = Macro(fIn);

    f.out[x, y | :] = fIn - TAU_INV * (fIn - Eq(rho, ux, uy));
}

gauge Hydro -> (real rho, real ux, real uy) {
    {rho, ux, uy} = Macro(f.in[x, y | :]);
}

node_classes {
    node_class BULK(InitTG, BGKCollStream, Hydro);
}

geometry {
    nodes[:,:] = BULK;
}
'''

with open('taylor_green.gelb', 'w') as f:
    f.write(GELB_SOURCE.lstrip('\n'))
show_gelb('taylor_green.gelb')

## 4. Compile, build, and run

The `gelb-run` wrapper does all three in one step: it invokes `gelbc` to emit Fortran, builds it with `gfortran`, and runs the executable. `-b cpu_fortran_serial` selects the Fortran backend; the default backend, `cpu_highway`, needs the Highway library of a source checkout (see the [README](https://github.com/dchirila/gelb#installation)). The `--write-every` flag controls how often the gauge samples the fields into NetCDF (every 20 steps here gives a smooth-looking time series).

In [ ]:
!gelb-run -b cpu_fortran_serial -n 2000 --write-every 20 -v taylor_green.gelb

## 5. Inspect the NetCDF output

The gauge block emits a standard NetCDF file. Every variable you wrote in `gauge Hydro(...)` becomes a separate NetCDF variable with shape `(time, y, x)` — Python's netCDF4 reverses the Fortran dimension order, so this is what you get directly.


In [ ]:
import netCDF4

ds = netCDF4.Dataset('taylor_green.nc')
print('Variables:', list(ds.variables.keys()))
print('Shape of ux:', ds.variables['ux'].shape, '  # (time, y, x)')
print('Number of snapshots:', ds.variables['ux'].shape[0])

## 6. Visualise the decay

### 6a. Initial and final velocity field

Each panel has its own colour scale: by the last step the pattern is the same, about a thousand times weaker.


In [ ]:
import matplotlib.pyplot as plt

ux = ds.variables['ux'][:]
uy = ds.variables['uy'][:]
write_every = 20

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, idx in [(axes[0], 0), (axes[1], ux.shape[0] - 1)]:
    vmax = abs(ux[idx]).max()
    im = ax.imshow(ux[idx], origin='lower', cmap='RdBu_r',
                   vmin=-vmax, vmax=vmax)
    ax.set_title(f'ux at step {idx * write_every}')
    ax.set_xlabel('x'); ax.set_ylabel('y')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

### 6b. Kinetic-energy decay vs. the analytical solution

For a Taylor-Green vortex, the total kinetic energy decays as

$$
KE(t) = KE(0) \cdot \exp(-4 \nu k^{2} t)
$$

with $k = 2\pi/N$: each velocity component decays as $\exp(-2\nu k^{2} t)$, since the wave vector has $k_x = k_y = k$, and the energy is quadratic in the velocity. Comparing the GeLB output against that curve is the standard validation check — any large deviation means the discrete BGK relaxation rate or the grid has a bug.


In [ ]:
import numpy as np

# KE per snapshot, summed over the lattice.
ke = 0.5 * (ux**2 + uy**2).sum(axis=(1, 2))

# Analytical: KE(t) ~ exp(-4*nu*k^2 * t), with t in LBM time units.
# We wrote every 20 steps, so the snapshot index needs a x20 factor.
N = 64
nu = 1.0 / 6.0
k = 2.0 * np.pi / N
write_every = 20
t = np.arange(ke.size) * write_every
ke_analytic = ke[0] * np.exp(-4.0 * nu * k**2 * t)

plt.figure(figsize=(7, 4))
plt.plot(t, ke,          label='GeLB simulation', linewidth=2)
plt.plot(t, ke_analytic, 'k--', label=r'analytical: $\exp(-4\nu k^{2} t)$')
plt.yscale('log')
plt.xlabel('time step')
plt.ylabel('total kinetic energy')
plt.title('Taylor-Green decay: GeLB vs. analytical')
plt.legend()
plt.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## Next steps

A few ways to extend this session:

- **Grid resolution**: at `N = 64` the measured decay rate matches $4\nu k^2$ to about 0.1 %. Change `const int N = 64;` (and `N` in the plotting cell) to 32 or 128 and rerun to see the error change with the grid.
- **Collision model**: compare with the MRT version, [`taylor_green_vortex_2d_mrt.gelb`](https://github.com/dchirila/gelb/blob/v1.0.0/examples/lbm/2d/taylor_green_vortex_2d_mrt.gelb).
- **Other 2-D examples**: [`examples/lbm/2d/`](https://github.com/dchirila/gelb/tree/v1.0.0/examples/lbm/2d) has the lid-driven cavity, Poiseuille, Couette, flow past a cylinder and Rayleigh-Bénard; [`examples/fd/2d/`](https://github.com/dchirila/gelb/tree/v1.0.0/examples/fd/2d) has finite-difference programs such as Gray-Scott.
- **3-D on a GPU**: open the companion [`gelb_gpu_3d.ipynb`](./gelb_gpu_3d.ipynb) notebook (requires a Colab GPU runtime).
- **Documentation**: the [GeLB User Tutorial](https://github.com/dchirila/gelb/blob/v1.0.0/docs/pdf/GeLB_User_Tutorial.pdf) (PDF) and the [repository](https://github.com/dchirila/gelb).

If GeLB is useful in academic work, please cite it: the [README](https://github.com/dchirila/gelb#using-gelb-in-your-work) gives the reference, and release 1.0.0 is archived as [doi:10.5281/zenodo.23592922](https://doi.org/10.5281/zenodo.23592922).